# Multi-anchor extension: nearest-anchor heuristic vs. a univariate KF + RTS smoother

[`ssp_07_extend_states_prior.ipynb`](./ssp_07_extend_states_prior.ipynb) walks through `extend_states_prior_nearest`, which fills
every undisclosed step of an anchored state with the **nearest**-anchor random-walk marginal,

$$a_{\text{obs}}[t] = a^{*}, \qquad P_{\text{obs}}[t] = P^{*} + |t - t^{*}|\,Q.$$

Its own docstring flags a limit: this is the exact marginal only for an **isolated channel with a
single anchor**. `ssp_07` shows that in that case the heuristic and the smoother are equal. This notebook
shows why the heuristic becomes a problem with **several** anchors: each step reads only its nearest anchor, so
the disclosures are never resolved into one continuous prior. It then builds out the principled alternative.

**The method under test (the proposed approach).** Since the state priors are not diffused across
states here — each disclosed state is its own input, coupled only later in the augmented-measurement
step — treat each state column as an *independent univariate time series* and run a standard
linear-Gaussian smoother over the same driftless random walk $x_t = x_{t-1} + w_t$,
$\mathrm{Var}(w_t) = Q$:

1. **Forward Kalman filter.** A disclosed anchor is fed in as an observation $(a^{*}, P^{*})$; an
   undisclosed step is a *missing observation*, so the filter just predicts — carry the mean, grow
   the variance by $Q$. The filter handles the `inf`/null steps natively; there's nothing special to
   do.
2. **Backward RTS smoother.** A second pass propagates each anchor's information *backward* too, so
   every step sees anchors on both sides.

For this linear-Gaussian model the RTS smoother is **exact** — it returns the true per-step posterior
marginal, no simulation smoother required. (A simulation smoother only earns its keep if we later
carry the nonlinear positivity reparam $\lambda = e^{k a}$ through the extension, or want *joint*
draws across time rather than per-step marginals — see the closing note.)

Unlike the nearest-anchor rule, this fuses *every* anchor in the channel through the shared Markov
chain — including, as we'll see, revising the anchors themselves.

In [ ]:
import jax

jax.config.update("jax_enable_x64", True)  # float64 for a clean diffuse-prior smoother

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from jax import numpy as jnp

from bunobee.models.ssp.plotting import (
    plot_prior_cross_sections,
    plot_prior_density,
    plot_prior_heatmap,
    plot_states,
)
from bunobee.models.ssp.prior import (
    disclosed_idx,
    extend_states_prior_nearest,
    extend_states_prior_smoothed,
)
from bunobee.simulation.ssp import construct_states_prior


## 1. A channel with several disclosed anchors

One state, hand-built so each anchor can carry its own mean and variance (unlike
`construct_states_prior`, which discloses one constant value per window). The disclosed values are
elasticity-like coefficients, so they sit in `[0, 1]`.

`multi_anchor` has three anchors at `t=5, 20, 34` with different means and precisions. The middle
anchor (`t=20`) is deliberately the **loose outlier** -- a somewhat higher value with large
variance -- so the two tighter neighbours should pull it back down once the channel is smoothed.

The disclosed values live in an editable `anchor_spec` at the top of the next cell -- tweak the
means / variances there and re-run to reconfigure the demo. (Changing the means/variances is safe;
changing the timesteps also means updating the hardcoded `rows` / `cross_t` picks in the later cells.)

In [ ]:
n_steps = 40
dates = pd.date_range("2024-01-07", periods=n_steps, freq="W")

# ---------------------------------------------------------------------------
# Anchor spec -- EDIT HERE to reconfigure the demo. One row per disclosed step:
#   (time_index, mean, variance)
#     mean     : elasticity-like coefficient, kept in [0, 1]
#     variance : disclosure tightness -- small = tight/confident, large = loose
# The `multi_anchor` middle row is the loose outlier (somewhat higher mean + large
# variance), so its two tight neighbours pull it back down once the channel is smoothed.
# ---------------------------------------------------------------------------
anchor_spec = {
    "multi_anchor": [(5, 0.30, 0.02), (20, 0.65, 0.15), (34, 0.35, 0.02)],
}

state_labels = list(anchor_spec)
n_states = len(state_labels)

a_obs = np.zeros((n_steps, n_states))
p_obs = np.full((n_steps, n_states), np.inf)
for s, label in enumerate(state_labels):
    for t, a_star, p_star in anchor_spec[label]:
        a_obs[t, s], p_obs[t, s] = a_star, p_star

base_prior = xr.Dataset(
    {
        "a_obs": (("time", "state"), a_obs),
        "P_obs": (("time", "state"), p_obs),
        "positivity": (("state",), np.zeros(n_states, dtype=bool)),
    },
    coords={"time": np.arange(n_steps), "state": state_labels},
)

anchors = disclosed_idx(base_prior)
print("disclosed steps:", anchors)
base_prior[["a_obs", "P_obs"]].isel(time=anchors).to_dataframe()

## 2. The univariate KF + RTS smoother, packaged as `extend_states_prior_smoothed`

The exact alternative ships in the package as
[`extend_states_prior_smoothed`](../../src/bunobee/models/ssp/prior.py) -- it reuses bunobee's own
diagonal 1-D Kalman filter and RTS smoother from `bunobee.models.ssp.kalman_1d`, the same code the
SSP models run in production:

- `kalman_filter_1d(...)` -- forward pass. Its optional **state-fusion** step precision-merges each
  disclosed anchor $(a^{*}, P^{*})$ into the carried state; steps left at `P_obs = inf` are no-ops
  (pure predict-through). That is exactly the *soft-observation* reading below.
- `kalman_rts_smoother_1d(at, Pt, sigma_q)` -- backward pass, returning smoothed means **and**
  marginal variances. It conditions on the fused anchors implicitly (they are already baked into
  `at`, `Pt`), so no extra arguments are needed.

**Driving it in "extension mode."** The filter is built around a scalar observation
$y_t = Z_t' \alpha_t + \varepsilon_t$ that couples the states through the design row $Z_t$. Here we
have no such $y$ -- only the disclosed anchors -- so `extend_states_prior_smoothed` passes a **zero
design matrix** $Z = 0$. That zeroes the Kalman gain, so the $y$-update contributes nothing and every
state evolves as an *independent* univariate random walk fed only by its own anchors. Two conventions
are handled internally: the filter takes a process-noise **standard deviation**, so it passes
`sigma_q = sqrt(Q)`; and the diffuse initial variance is a large finite `P0_diffuse` (default `1e8`,
comfortable in float64) standing in for "no prior information."

**Why fusion, not injection.** The anchor enters as a *soft observation* -- a measurement with noise
variance $R = P^{*}$ -- not a hard pin. When the incoming prediction is still diffuse (first anchor
reached) the posterior collapses to $(a^{*}, P^{*})$ exactly, so an isolated anchor behaves like the
heuristic; but when information is already flowing in from another anchor, the two are *fused* by
inverse-variance weighting. That fusion -- $P^{*}$ read as a stated *uncertainty* rather than a hard
value -- is the whole reason the multi-anchor result departs from the nearest-anchor rule.

(One consequence of the diffuse `P0`: inside the filter a state with *no* anchor comes back
diffuse-but-finite; `extend_states_prior_smoothed` resets those columns to `inf`, matching
`extend_states_prior_nearest`. The state below is anchored, so it doesn't arise.)

## 3. Heuristic vs. smoother, side by side

Run both extensions on the same base prior and line up a handful of representative steps: each
anchor, the steps either side of each nearest-anchor switch, and a couple of steps outside the anchor span.
The `reference` columns below are the KF + RTS smoother from section 2.

In [ ]:
Q = 0.02  # per-state process variance sigma_q**2, same default as ssp_07_extend_states_prior.ipynb

heuristic = extend_states_prior_nearest(base_prior, Q)
reference = extend_states_prior_smoothed(base_prior, Q)  # exact KF + RTS marginal

s = state_labels.index("multi_anchor")
rows = [0, 5, 12, 13, 20, 26, 27, 34, 39]  # anchors (5, 20, 34), the steps either side of each switch, both tails
comparison = pd.DataFrame(
    {
        "a_heuristic": heuristic["a_obs"].values[rows, s],
        "P_heuristic": heuristic["P_obs"].values[rows, s],
        "a_reference": reference["a_obs"].values[rows, s],
        "P_reference": reference["P_obs"].values[rows, s],
    },
    index=pd.Index(rows, name="t"),
)
comparison["d_mean"] = comparison["a_reference"] - comparison["a_heuristic"]
comparison["P_ratio (heuristic / reference)"] = comparison["P_heuristic"] / comparison["P_reference"]
comparison.round(4)

Even at `t=20` -- an anchor itself -- the reference revises the disclosed value: because it treats
`P*` as genuine observation noise rather than a hard pin, the tighter anchors at `t=5` and `t=34` pull
the smoothed estimate away from the raw disclosure. `extend_states_prior_nearest` freezes every anchor exactly
(by construction, see `test_anchor_entry_preserved_exactly` in
`tests/test_extend_states_prior.py`), so it cannot reproduce that. Between anchors, the reference's
`P_ratio` is consistently `>= 1`: fusing two directions of information is never less informative than
picking the single nearer anchor.

### One channel, three bands

The same `multi_anchor` channel drawn three ways, each as a 90% band on a shared y-axis:

1. **`ssp_07` sampling** -- the recipe from `ssp_07_extend_states_prior.ipynb`: extend with
   `extend_states_prior_nearest`, draw Monte Carlo samples from each step's marginal, and take the 5% / 95%
   quantiles.
2. **Nearest-anchor heuristic** -- the same extended prior, with the band read directly from
   $a_{\text{obs}} \pm 1.645\,\sqrt{P_{\text{obs}}}$.
3. **KF + RTS smoother** -- the band read directly from the smoothed $(a_{\text{obs}}, P_{\text{obs}})$.

In [ ]:
def prior_samples(ssp_prior, n_draws=2000, seed=0):
    """Draw Monte Carlo samples from a prior's per-step Gaussian marginal.

    Parameters
    ----------
    ssp_prior : xr.Dataset
        Prior with ``a_obs`` / ``P_obs`` over ``(time, state)``.
    n_draws : int, optional
        Number of Monte Carlo draws, by default 2000.
    seed : int, optional
        RNG seed, by default 0.

    Returns
    -------
    np.ndarray, shape (n_draws, n_steps, n_states)
        Draws from ``N(a_obs, P_obs)`` per step, shaped for ``plot_states``.
        Undisclosed (``inf``-variance) steps are set to ``NaN`` so the ribbon
        shows a gap there.
    """
    rng = np.random.default_rng(seed)
    a = np.asarray(ssp_prior["a_obs"].values, dtype=float)
    p = np.asarray(ssp_prior["P_obs"].values, dtype=float)
    draws = a[None] + np.sqrt(p)[None] * rng.standard_normal((n_draws, *a.shape))
    draws[:, ~np.isfinite(p)] = np.nan
    return draws


def exact_band(ssp_prior, s, z=1.645):
    """Return the exact central 90% band of one state's per-step Gaussian marginal.

    Parameters
    ----------
    ssp_prior : xr.Dataset
        Prior with ``a_obs`` / ``P_obs`` over ``(time, state)``.
    s : int
        Index of the state to read.
    z : float, optional
        Standard-normal quantile setting the band width, by default 1.645 (90%).

    Returns
    -------
    tuple of np.ndarray
        ``(lower, centre, upper)``, each of shape ``(n_steps,)``.
    """
    mean = ssp_prior["a_obs"].values[:, s]
    half_width = z * np.sqrt(ssp_prior["P_obs"].values[:, s])
    return mean - half_width, mean, mean + half_width


steps = np.arange(n_steps)
sampled_lo, sampled_mid, sampled_hi = np.quantile(prior_samples(heuristic)[:, :, s], [0.05, 0.5, 0.95], axis=0)
bands = [
    ("(i) ssp_07 sampling of the heuristic", (sampled_lo, sampled_mid, sampled_hi), "steelblue"),
    ("(ii) nearest-anchor heuristic, exact", exact_band(heuristic, s), "steelblue"),
    ("(iii) KF + RTS smoother, exact", exact_band(reference, s), "darkorange"),
]

fig, axes = plt.subplots(1, 3, figsize=(14, 3.8), sharey=True)
anchor_steps = anchors[np.isfinite(base_prior["P_obs"].values[anchors, s])]
for ax, (name, (lo, mid, hi), color) in zip(axes, bands):
    ax.fill_between(steps, lo, hi, color=color, alpha=0.25, linewidth=0, label="90% band")
    ax.plot(steps, mid, color=color, lw=2, label="centre")
    ax.scatter(
        anchor_steps,
        base_prior["a_obs"].values[anchor_steps, s],
        color="crimson",
        marker="x",
        s=50,
        zorder=3,
        label="raw anchor",
    )
    ax.set_title(name, fontsize=10)
    ax.set_xlabel("t")
    ax.grid(alpha=0.2)
axes[0].set_ylabel("multi_anchor state value")
axes[0].legend(fontsize=8, loc="upper left")
fig.suptitle(f"90% prior band of one channel under each method (Q={Q})", fontsize=11)
plt.tight_layout()
plt.show()

width_gap = np.abs((sampled_hi - sampled_lo) - (bands[1][1][2] - bands[1][1][0])).max()
print(f"max |band width (i) - band width (ii)| = {width_gap:.3f}")
print(f"largest step-to-step jump in mean, heuristic: {np.abs(np.diff(heuristic['a_obs'].values[:, s])).max():.3f}")
print(f"largest step-to-step jump in mean, smoother:  {np.abs(np.diff(reference['a_obs'].values[:, s])).max():.3f}")

Panels (i) and (ii) are one method drawn two ways: `ssp_07` samples the nearest-anchor prior, so its ribbon
differs from the exact band only by Monte Carlo noise, which is the small gap printed above.

The problem is (ii) against (iii). The heuristic never resolves the three disclosures against each other. Each
step reads one anchor and ignores the other two, so the prior is three separate pieces. Its mean jumps from
`0.30` to `0.65` between `t=12` and `t=13`, where the nearest anchor switches, and the variance jumps with it,
from `0.16` to `0.29`. A random walk has no such break, so it comes from the rule and not from the model. The
smoother fuses all three anchors into one prior, so its mean moves continuously through them, its largest
step-to-step move is a small fraction of the heuristic's jump, and its band is narrower at every step.

The rest of this notebook focuses on the smoother.

## 4. The final smoothed distribution

The two-pass smoother returns a full Gaussian marginal $\mathcal{N}(a_{\text{obs}}[t],
P_{\text{obs}}[t])$ at every step -- that *is* the final distribution the extension produces. Two
views below:

- **Density field** -- the smoother's posterior density across the whole horizon (bright = high
  density). The mean threads through the anchors and the band pinches to each disclosure, widening
  between them.
- **Cross-sections** -- the same distribution sliced at each anchor as a bell curve, with the raw
  disclosure $\mathcal{N}(a^{*}, P^{*})$ overlaid so smoothing's revision of each anchor reads as a density.

In [ ]:
# Pass the smoothed `reference` prior for the density field and the raw `base_prior` for the anchor
# markers (every step of `reference` is finite, so its own disclosures no longer isolate the raw anchors).
fig, axes = plot_prior_density(reference, anchors=base_prior, title="final smoothed density")
plt.show()

In [ ]:
fig, axes = plot_prior_cross_sections(
    {"raw disclosure": base_prior, "smoothed (final)": reference},
    state="multi_anchor",
    times=[5, 20, 34],  # tight anchor, loose anchor, tight anchor
    colors={"raw disclosure": "crimson", "smoothed (final)": "darkorange"},
    title="Smoothing's revision of each disclosed anchor",
)
plt.show()

The `t=20` panel is the headline: the raw disclosure sits above its neighbours at `0.65` with a large `P*`, and
the smoother returns a narrower bell pulled down to about `0.49`, toward the tight neighbours at `t=5` (`0.30`)
and `t=34` (`0.35`). At those tight anchors the smoothed bell nearly coincides with the disclosure -- the
single-anchor limit in miniature.

## 5. A realistic multi-channel prior, end to end

Sections 1-4 ran on a hand-built one-channel prior so every anchor could carry its own mean and
variance. Nothing about `extend_states_prior_smoothed` is confined to that toy: it runs a **single**
vectorised `kalman_filter_1d` with a zero design matrix $Z = 0$, so the state columns are exactly
decoupled and a whole `construct_states_prior` output extends in one call.

This section uses the same layout as
[`ssp_07_extend_states_prior.ipynb`](./ssp_07_extend_states_prior.ipynb) -- a weekly 52-step, 4-state prior
over a `level` plus three media regressors -- but discloses it over several windows, and runs the smoother
over it with a per-state process-variance vector `Q`. The `level` state is disclosed nowhere (its variance stays
`inf`), so it also exercises the "no anchor -> stays undisclosed" path.

In [ ]:
regressors = ["tv", "search", "social"]
mc_labels = ["level", *regressors]
mc_n_states = len(mc_labels)
mc_n_steps = 52

mc_prior = construct_states_prior(
    n_steps=mc_n_steps,
    n_states=mc_n_states,
    true_states=jnp.array([0.0, 0.8, 0.5, 0.3]),  # level entry ignored (its variance stays inf)
    regressors=regressors,
    n_periods=3,
    n_points=4,
    seed=7,
    obs_scale=0.1,
)

mc_dates = pd.date_range("2024-01-01", periods=mc_n_steps, freq="W")
mc_anchors = disclosed_idx(mc_prior)
print("disclosed anchor steps:", mc_anchors)
print("finite fraction (base):", np.isfinite(mc_prior["P_obs"].values).mean().round(3))

Each regressor is disclosed over the same handful of windows, so every anchored channel here is a
genuinely **multi-anchor** one -- the regime section 3 showed the nearest-anchor heuristic drifting in.

`extend_states_prior_smoothed` accepts a length-`n_states` `Q` instead of a scalar, giving each channel its own
random-walk slope: how fast its variance opens up away from an anchor. Below, `search` is a slow,
well-behaved channel (`Q = 0.005`) and `social` a volatile one (`Q = 0.05`).

In [ ]:
# One process variance per state; the level entry is inert (no anchor to extend from).
Q_vec = np.array([0.02, 0.02, 0.005, 0.05])  # level, tv, search, social

mc_smoothed = extend_states_prior_smoothed(mc_prior, Q_vec)

print("finite fraction (smoothed):", np.isfinite(mc_smoothed["P_obs"].values).mean().round(3))
print("level stays fully undisclosed:", bool(np.all(np.isinf(mc_smoothed["P_obs"].values[:, 0]))))

pd.DataFrame(
    {
        "Q": Q_vec,
        "max P_obs (smoothed)": mc_smoothed["P_obs"].values.max(axis=0),
    },
    index=pd.Index(mc_labels, name="state"),
).round(4)

The widest step of each anchored channel follows its own `Q`: smallest for `search`, largest for `social`, while
`level` stays at `inf`. The heatmap below is the smoothed prior itself: every anchored row is filled edge to
edge, and the unanchored `level` row stays masked (grey).

In [ ]:
fig, axes = plot_prior_heatmap(
    mc_smoothed,
    quantity="both",
    dates=mc_dates.values,
    title="smoothed extension of the multi-channel prior (mean & variance)",
)
plt.show()

### The smoothed prior against its anchors

Here we reuse the `prior_samples` helper from section 3 to draw from each prior's per-step marginal
$\mathcal{N}(a_{\text{obs}}, P_{\text{obs}})$, so the ribbons are quantiles over draws. Undisclosed steps (`inf`
variance) become `NaN` and leave a gap, so the **anchors only** ribbon shows just the disclosure windows and the
`level` panel stays empty.

In [ ]:
# `prior_samples` is defined in section 3; reuse it to draw from each prior's per-step marginal
# N(a_obs, P_obs). Undisclosed (inf-variance) steps come back as NaN, leaving a ribbon gap -- so the
# "anchors only" ribbon shows only the disclosure windows and the `level` panel stays empty.
mc_posterior = {
    "anchors only": prior_samples(mc_prior),
    "in-house KF + RTS smoother": prior_samples(mc_smoothed),
}

fig, axes = plot_states(
    mc_posterior,
    mc_dates.values,
    mc_labels,
    states_key=list(mc_posterior),
    obs_idx=mc_anchors,
    a_obs=mc_prior["a_obs"].values,
    P_obs=mc_prior["P_obs"].values,
    title=f"Multi-channel smoothed extension of a construct_states_prior output (Q={Q_vec.tolist()})",
    n_cols=2,
    colors={
        "anchors only": "darkgreen",
        "in-house KF + RTS smoother": "darkorange",
    },
)
plt.show()

Every regressor panel tells the same story: the smoother's ribbon pinches at each disclosure window and opens
smoothly between them, at each channel's own rate -- `social` (`Q = 0.05`) has ten times the process variance of
`search` (`Q = 0.005`). Red crosses mark the raw disclosed anchors; the `level` panel is empty because an
unanchored channel has nothing to extend.

### The channels really are decoupled

The zero design matrix is what licenses running all four channels through one filter, so it is worth
checking rather than asserting: extending each column **alone**, as a one-state prior, must reproduce
the joint four-state run exactly -- not merely to a tolerance.

In [ ]:
checks = []
for i, label in enumerate(mc_labels):
    solo = extend_states_prior_smoothed(mc_prior.isel(state=[i]), Q_vec[i])
    checks.append(
        {
            "state": label,
            "a_obs identical": np.array_equal(solo["a_obs"].values[:, 0], mc_smoothed["a_obs"].values[:, i]),
            "P_obs identical": np.array_equal(solo["P_obs"].values[:, 0], mc_smoothed["P_obs"].values[:, i]),
            "P_obs all inf": bool(np.all(np.isinf(mc_smoothed["P_obs"].values[:, i]))),
        }
    )

pd.DataFrame(checks).set_index("state")

Bit-for-bit on every channel, and `level` -- the only column with no anchor -- comes back at
`P_obs = inf` throughout, exactly as `extend_states_prior_nearest` leaves it. The vectorised call is
therefore only a performance convenience: it computes what four independent univariate smoothers
would, which is the whole point of driving the filter with $Z = 0$.

## Takeaways

- **Single anchor per channel:** the heuristic and the smoother are equal; `ssp_07` demonstrates it.
- **The heuristic does not resolve several anchors into one prior:** each step reads only its nearest anchor, so
  the mean and variance jump wherever that anchor switches. The break comes from the rule, not from the random walk.
- **Sampling is not a third method:** the Monte Carlo ribbon in `ssp_07` is a rendering of the nearest-anchor
  prior, so it matches the heuristic's exact band up to sampling noise. The choice that matters is heuristic
  vs. smoother.
- **Two or more anchors:** the heuristic and the smoother diverge on both moments. The heuristic
  hard-switches to whichever anchor is nearer and freezes each anchor's disclosed value exactly; the
  smoother fuses every anchor's propagated estimate by inverse-variance weighting, which (a) blends
  the mean between anchors instead of jumping, (b) tightens the variance versus either one-sided
  propagation alone, and (c) revises the anchors themselves once more than one is disclosed.
- **Many channels at once:** section 5 runs the smoother over a full `construct_states_prior`
  output (52 weekly steps, a `level` plus three regressors) with a per-state `Q` vector. Because the
  smoother drives the filter with $Z = 0$, the columns stay exactly decoupled -- extending each one
  alone reproduces the joint run bit-for-bit -- and the unanchored `level` comes back at `P_obs = inf`.
- **The proposed method works and is exact here.** Treating each state prior as an independent
  univariate series, filtering forward (anchors = observations, nulls = predict-only) and smoothing
  backward is well-posed and, for this linear-Gaussian random walk, returns the *exact* posterior
  marginal. No simulation smoother is needed for this step.
- **When a simulation smoother would be needed instead:** (1) if the nonlinear positivity reparam
  $\lambda = e^{k a}$ is carried through the extension so the model is no longer linear-Gaussian, or
  (2) if we need *joint* posterior draws across time (correlated sample paths) rather than the
  per-step marginals the RTS pass returns. Neither applies to producing $(a_{\text{obs}},
  P_{\text{obs}})$ marginals, so RTS is the right tool here.
- **Practical read:** for a genuinely multi-anchor channel, `extend_states_prior_nearest`'s output is a
  conservative, discontinuous approximation -- correctly shaped, but neither as tight nor as smooth as
  a real smoother pass. That's consistent with the function's own framing: feed the extension into the
  augmented-measurement step as a prior, not as a final posterior. Wiring the in-house
  `kalman_filter_1d` + `kalman_rts_smoother_1d` in (as `extend_states_prior_smoothed` does here) is a candidate
  upgrade if multi-anchor channels are common in practice.